# Mini Lab: Logistic Regression & SVM

### Name: Josh Samuel
### DS 7331

In [1]:
# Load libraries
import time

import pandas as pd
from clean import load_raw, clean_mushrooms
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             classification_report, f1_score, confusion_matrix)

In [2]:
# Load clean dataset
df = clean_mushrooms(load_raw())
X = df.drop(columns="class") # Predictors (every column but class)
y = (df["class"] == "poisonous").astype(int)   # Target variable (1 if poisonous, 0 if edible)

In [3]:
# Number of NA values in each column
df.isna().sum()

cap-diameter            0
cap-shape               0
cap-surface             0
cap-color               0
does-bruise-or-bleed    0
gill-attachment         0
gill-spacing            0
gill-color              0
stem-height             0
stem-width              0
stem-root               0
stem-surface            0
stem-color              0
veil-type               0
veil-color              0
has-ring                0
ring-type               0
spore-print-color       0
habitat                 0
season                  0
class                   0
has-stem                0
size-score              0
dtype: int64

## Question 1
Create a logistic regression model and a support vector machine model for the
classification task involved with your dataset. Assess how well each model performs (use
80/20 training/testing split for your data). Adjust parameters of the models to make them more
accurate. If your dataset size requires the use of stochastic gradient descent, then linear kernel
only is fine to use

#### Baseline Logistic Regression Model

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,test_size=0.20, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(48738, 22) (12185, 22) (48738,) (12185,)


In [5]:
# Split Feature Columns
num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes(exclude="number").columns

In [6]:
# Preprocessing Step
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

In [7]:
# Baseline logistic regression model
baseline_log = Pipeline([
    ("preprocessing", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

start_time = time.time()
baseline_log.fit(X_train, y_train)
baseline_log_training_time = time.time() - start_time

y_pred_log = baseline_log.predict(X_test)
print(f"Training time: {baseline_log_training_time:.2f} seconds")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_log))
print("\nClassification report:\n", classification_report(
    y_test, y_pred_log, target_names=["edible", "poisonous"]))

Training time: 0.40 seconds
Accuracy: 0.8640131308986458
Confusion matrix:
 [[4593  843]
 [ 814 5935]]

Classification report:
               precision    recall  f1-score   support

      edible       0.85      0.84      0.85      5436
   poisonous       0.88      0.88      0.88      6749

    accuracy                           0.86     12185
   macro avg       0.86      0.86      0.86     12185
weighted avg       0.86      0.86      0.86     12185



#### Baseline SVM Model

In [8]:
# Baseline SVM model (RBF kernel)
svm_baseline = Pipeline([
    ("preprocessing", preprocess),
    ("model", SVC(kernel="rbf", random_state=42))
])

start_time = time.time()
svm_baseline.fit(X_train, y_train)
baseline_svm_training_time = time.time() - start_time

y_pred_svm = svm_baseline.predict(X_test)
print(f"Training time: {baseline_svm_training_time:.2f} seconds")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("\nClassification report:\n", classification_report(
    y_test, y_pred_svm, target_names=["edible", "poisonous"]))

Training time: 10.47 seconds
Accuracy: 0.9996717275338531
Confusion matrix:
 [[5432    4]
 [   0 6749]]

Classification report:
               precision    recall  f1-score   support

      edible       1.00      1.00      1.00      5436
   poisonous       1.00      1.00      1.00      6749

    accuracy                           1.00     12185
   macro avg       1.00      1.00      1.00     12185
weighted avg       1.00      1.00      1.00     12185



#### Tuning - Logistic Regression

In [9]:
grid_logistic = GridSearchCV(
    baseline_log,
    param_grid={"model__C": [0.01, 0.1, 1, 10, 100], "model__class_weight": [None, "balanced"]},
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train and time the model
start_time = time.time()
grid_logistic.fit(X_train, y_train)
grid_logistic_time = time.time() - start_time

print(f"Grid search time: {grid_logistic_time:.2f} seconds")
print("Best parameters:", grid_logistic.best_params_)
print("Best CV accuracy:", grid_logistic.best_score_)

Grid search time: 5.61 seconds
Best parameters: {'model__C': 100, 'model__class_weight': 'balanced'}
Best CV accuracy: 0.8686652130666218


#### Tuning - SVM

In [ ]:
# Create a representative 10,000-row subset for faster tuning
X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=10000,
    stratify=y_train,
    random_state=42
)

# Parameters to test
svm_param_grid = {
    "model__C": [0.1, 1, 10], 
    hwy # smaller C is more willing to misclassify points but keep a wider margin, larger C is less willing to misclassify points even if margin is narrower
    "model__class_weight": [None, "balanced"]
}

# Tune on the smaller training subset
grid_svm_small = GridSearchCV(
    estimator=svm_baseline,
    param_grid=svm_param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

start_time = time.time()
grid_svm_small.fit(X_tune, y_tune)
tuning_time = time.time() - start_time

print(f"Tuning time: {tuning_time:.2f} seconds")
print("Best parameters:", grid_svm_small.best_params_)
print("Best cross-validation accuracy:", grid_svm_small.best_score_)

# Retrieve the best configuration and refit it on all training data
best_svm = grid_svm_small.best_estimator_

start_time = time.time()
best_svm.fit(X_train, y_train)
final_training_time = time.time() - start_time

# Test-set predictions
y_pred_svm_tuned = best_svm.predict(X_test)

# Test-set evaluation
print(f"\nFinal training time: {final_training_time:.2f} seconds")
print("Test accuracy:", accuracy_score(y_test, y_pred_svm_tuned))

print(
    "\nConfusion matrix:\n",
    confusion_matrix(y_test, y_pred_svm_tuned))

print(
    "\nClassification report:\n", classification_report(y_test, y_pred_svm_tuned,
    target_names=["edible", "poisonous"]))

Tuning time: 6.55 seconds
Best parameters: {'model__C': 10, 'model__class_weight': None}
Best cross-validation accuracy: 0.9996000199900014

Final training time: 4.77 seconds
Test accuracy: 1.0

Confusion matrix:
 [[5436    0]
 [   0 6749]]

Classification report:
               precision    recall  f1-score   support

      edible       1.00      1.00      1.00      5436
   poisonous       1.00      1.00      1.00      6749

    accuracy                           1.00     12185
   macro avg       1.00      1.00      1.00     12185
weighted avg       1.00      1.00      1.00     12185



In [11]:
# Save each model's predictions once, then compare them
models = [
    ("Baseline Logistic Regression", baseline_log, baseline_log_training_time),
    ("Baseline SVM", svm_baseline, baseline_svm_training_time),
    ("Tuned Logistic Regression", grid_logistic, grid_logistic_time),
    ("Tuned SVM", best_svm, tuning_time + final_training_time)
]

rows = []
for name, model, seconds in models:
    predictions = model.predict(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Poisonous Precision": precision_score(y_test, predictions),
        "Poisonous Recall": recall_score(y_test, predictions),
        "Poisonous F1": f1_score(y_test, predictions),
        "Total Training / Tuning Time (sec)": seconds
    })

comparison_table = pd.DataFrame(rows)
comparison_table.round(3)

,Model,Accuracy,Poisonous Precision,Poisonous Recall,Poisonous F1,Total Training / Tuning Time (sec)
0,Baseline Logistic Regression,0.864,0.876,0.879,0.878,0.402
1,Baseline SVM,1.000,0.999,1.000,1.000,10.472
2,Tuned Logistic Regression,0.870,0.899,0.861,0.880,5.615
3,Tuned SVM,1.000,1.000,1.000,1.000,9.824


- **Accuracy:** Percentage of all mushrooms classified correctly.
- **Poisonous precision:** Of the mushrooms predicted poisonous, the percentage that actually were poisonous.
- **Poisonous recall:** Of the mushrooms that actually were poisonous, the percentage identified as poisonous. A missed poisonous mushroom would be predicted edible.
- **Poisonous F1:** One score that combines poisonous precision and recall.
- **Total training / tuning time:** For baseline models, time to fit one model. For tuned models, grid search time plus the final full-training fit. The SVM grid search used a 10,000-row training subset, while the logistic regression grid search used all training rows, so these times reflect different amounts of work.

## Question 2
Discuss the advantages of each model for each classification task. Does one type
of model offer superior performance over another in terms of prediction accuracy? In terms of
training time or efficiency? Explain in detail.

Logistic regression was fast and gave us coefficients we could interpret. Its baseline test accuracy was 86.4%, increasing to 87.0% after tuning. The RBF SVM could learn a more flexible boundary between edible and poisonous mushrooms. It made four mistakes before tuning and none after tuning on the 12,185 test mushrooms. In both cases, it identified every poisonous mushroom in this test set.

Logistic regression took about 0.402 seconds for a single baseline fit, compared with about 10.47 seconds for the baseline SVM. The tuned models took longer because grid search fit multiple models. Overall, the tuned SVM performed best on this test set, while logistic regression was faster to train and easier to explain. A perfect test result does not guarantee perfect predictions on new mushrooms.

## Question 3
Use the weights from logistic regression to interpret the importance of different
features for each classification task. Explain your interpretation in detail. Why do you think
some variables are more important?

In [12]:
# Extract the best logistic-regression pipeline from GridSearchCV
best_logistic = grid_logistic.best_estimator_

# Extract preprocessing and logistic-regression steps
preprocessor = best_logistic.named_steps["preprocessing"]
log_model = best_logistic.named_steps["model"]

# Match coefficients to transformed feature names
feature_names = preprocessor.get_feature_names_out()

coefficient_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": log_model.coef_[0]
})

coefficient_table["Absolute Importance"] = (
    coefficient_table["Coefficient"].abs()
)

# Most important features overall
coefficient_table.sort_values(
    "Absolute Importance",
    ascending=False
).head(15)

,Feature,Coefficient,Absolute Importance
96,cat__veil-color_yellow,-16.394261,16.394261
107,cat__ring-type_zone,15.757089,15.757089
62,cat__stem-root_club,13.050520,13.050520
109,cat__spore-print-color_black,12.317945,12.317945
104,cat__ring-type_movable,-11.876325,11.876325
93,cat__veil-color_purple,11.178329,11.178329
89,cat__veil-type_universal,10.901161,10.901161
68,cat__stem-surface_grooves,10.720699,10.720699
95,cat__veil-color_white,-10.388112,10.388112
64,cat__stem-root_rooted,8.377090,8.377090


### Interpretation

We used the tuned logistic regression model to see which mushroom characteristics had the strongest weights. Our target is **1 = poisonous**, so a positive coefficient pushes the prediction toward poisonous; a negative coefficient pushes it toward edible.

Some of the largest weights were yellow veil color (−16.39), zone ring type (+15.76), club-shaped stem root (+13.05), and black spore print (+12.32). For example, the model associated a zone ring with poisonous mushrooms and a yellow veil with edible mushrooms, after accounting for the other features.

These categories may have large weights because their patterns help the model distinguish the two classes in this dataset. Each coefficient describes a *specific category*, not an entire characteristic such as ring type. A large weight does not mean that a feature causes toxicity or that we can use that feature alone to identify a mushroom. The numerical features were scaled and the categorical features were encoded differently, so the absolute coefficients are only a rough guide to importance.

## Question 4
Look at the chosen support vectors for the classification task. Do these provide
any insight into the data? Explain.

In [13]:
# Extract the fitted SVM from the tuned SVM pipeline
svm_model = best_svm.named_steps["model"]

# Number of support vectors
print("Total support vectors:", len(svm_model.support_))
print("Support vectors by class:", svm_model.n_support_)
print("Class order:", svm_model.classes_)

# Percentage of training observations used as support vectors
support_vector_pct = len(svm_model.support_) / len(X_train) * 100
print(f"Percent of training data used as support vectors: {support_vector_pct:.2f}%")

# View the original, unencoded rows that became support vectors
support_vector_rows = X_train.iloc[svm_model.support_].copy()
support_vector_rows["Actual Class"] = y_train.iloc[svm_model.support_].map(
    {0: "edible", 1: "poisonous"}
)

support_vector_rows.head(10)

Total support vectors: 1057
Support vectors by class: [504 553]
Class order: [0 1]
Percent of training data used as support vectors: 2.17%


,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,stem-width,...,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season,has-stem,size-score,Actual Class
20479,3.49,convex,Missing,red,no,adnate,close,orange,4.10,6.39,...,Missing,Missing,none,none,Missing,meadows,summer,yes,8.229,edible
30640,11.98,sunken,sticky,pink,yes,decurrent,close,orange,5.38,16.66,...,Missing,Missing,none,none,Missing,woods,autumn,yes,19.026,edible
20495,3.17,convex,smooth,pink,no,adnate,close,orange,4.27,6.01,...,Missing,Missing,none,none,Missing,grasses,autumn,yes,8.041,edible
11460,1.59,conical,grooves,brown,no,adnate,Missing,white,5.66,1.58,...,Missing,Missing,none,none,Missing,woods,autumn,yes,7.408,edible
51848,14.05,convex,Missing,red,no,pores,Missing,brown,13.95,33.45,...,Missing,Missing,none,none,Missing,woods,autumn,yes,31.345,edible
31311,8.81,convex,smooth,brown,no,free,close,pink,8.12,9.77,...,Missing,Missing,none,none,pink,woods,summer,yes,17.907,edible
12373,1.50,convex,smooth,brown,no,decurrent,Missing,white,5.29,1.88,...,Missing,Missing,none,none,Missing,grasses,autumn,yes,6.978,edible
26175,8.71,flat,smooth,purple,no,adnexed,Missing,brown,5.35,18.77,...,Missing,Missing,none,none,Missing,woods,autumn,yes,15.937,edible
45747,1.04,spherical,grooves,white,no,Missing,Missing,gray,3.60,1.72,...,Missing,Missing,none,none,Missing,woods,spring,yes,4.812,edible
33856,4.67,sunken,leathery,white,no,decurrent,Missing,white,3.21,7.64,...,Missing,Missing,none,none,Missing,grasses,summer,yes,8.644,edible


### Interpretation

The tuned SVM used **1,057 of the 48,738 training mushrooms** as support vectors, or **2.17%**. Of these, **504 were edible** and **553 were poisonous**.

Support vectors are training examples that help set the boundary between the classes. Since only a small portion of the training data were support vectors, many other examples did not directly set the boundary. Support vectors may include mushrooms near the boundary or on the wrong side of its margin; they are not necessarily mistakes.

The first ten original rows do not show an obvious shared characteristic. We would need to examine the support vectors more systematically before saying which combinations of features are hardest to separate. The tuned SVM made no errors on this test set, but that does not guarantee it will classify every new mushroom correctly.